# From clicks to settlement: walkthrough

This notebook reads the outputs produced by `make features models` and re-plots the key results. Run the pipeline first. Nothing here hits the network.

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
D = Path('../data/derived'); M = D / 'models'
daily = pd.read_csv(D / 'daily.csv', parse_dates=['date']).set_index('date')
monthly = pd.read_csv(D / 'monthly.csv', parse_dates=['month']).set_index('month')
daily.tail(3)

## 1. Raw activity vs value-weighted activity

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
daily['active_accounts'].rolling(30).mean().plot(ax=axes[0], logy=True, title='Daily active accounts (30d mean, log)')
monthly['intents_volume_usd'].plot(ax=axes[1], kind='bar', title='Intents volume per month (USD)'); axes[1].set_xticklabels([t.strftime('%Y-%m') if i % 6 == 0 else '' for i, t in enumerate(monthly.index)])
daily['usd_per_txn'].rolling(30).mean().plot(ax=axes[2], logy=True, title='USD per transaction (30d mean, log)')
plt.tight_layout()

## 2. Structural breaks and correlations (L3)

In [ ]:
breaks = json.loads((M / 'breaks.json').read_text())
{k: v['breaks'] for k, v in breaks.items()}

In [ ]:
pd.read_csv(M / 'yearly_corr.csv').set_index('year').round(2)

In [ ]:
pd.DataFrame(json.loads((M / 'var_granger.json').read_text())).set_index('metric')[['n_weeks', 'lag_aic', 'usage_granger_causes_price', 'price_granger_causes_usage']]

## 3. Signal tests and scenarios (L4)

In [ ]:
ic = pd.DataFrame(json.loads((M / 'signal_ic.json').read_text()))
ic[ic.horizon_weeks == 12].pivot_table(index='metric', columns='benchmark', values='spearman_ic').round(2)

In [ ]:
sc = json.loads((M / 'scenarios.json').read_text()); sc['inputs'], sc['price_to_fees']

## 4. Quality-adjusted accounts (L1) and Intents microstructure (L2)

These cells only run after `make bq` has produced the BigQuery extracts.

In [ ]:
if (M / 'qaa_monthly.csv').exists():
    q = pd.read_csv(M / 'qaa_monthly.csv', parse_dates=['month']).set_index('month')
    q[['active_accounts_bq', 'qaa']].plot(logy=True, title='Active accounts vs QAA')
    display(pd.read_csv(M / 'qaa_clusters.csv'))
else:
    print(json.loads((M / 'qaa_status.json').read_text()))

In [ ]:
if (M / 'intents_monthly.csv').exists():
    im = pd.read_csv(M / 'intents_monthly.csv', parse_dates=['month']).set_index('month')
    im[['hhi_solver', 'top1_solver_share']].plot(title='Solver concentration')
    display(im.tail())
else:
    print(json.loads((M / 'intents_status.json').read_text()))